# 08_GenAI_Copilot

In [ ]:
!pip install python-dotenv



In [ ]:
!pip install groq

In [16]:
from dotenv import load_dotenv
import os

from google import genai
import pandas as pd

load_dotenv()

df = pd.read_csv(
    "../data/processed/risk_scored_dataset.csv"
)

df.head()

,Agmt Id,Cust Age,Cust Gender,Cust Cibil Score,Cust Employment Type,Cust Net Salary,Coborrower Flag,App Score Risk,Agmt Date,Seizure Date,...,Recovery_Efficiency_Index,LGD,LGD_pct,Depreciation_Pct,Asset_Age_Group,Residual_Value_Forecast,Residual_Loss,Residual_Risk_Score,Risk_Band,Profitability_Score
0,ASSET_1,29,F,-1,NREGI,30000,N,LOW RISK,2023-12-27,2025-05-23,...,78.834107,9934.0,21.165893,54.320988,1-2 Years,31061.247508,49938.752492,61.652781,High,49.149826
1,ASSET_2,24,M,-1,NREGI,60000,N,LOW RISK,2025-11-20,2026-05-31,...,65.745448,29177.0,34.254552,47.565543,0-1 Year,49192.500708,57607.499292,53.939606,Low,61.601845
2,ASSET_3,32,F,763,AGR,29500,N,LOW RISK,2024-05-16,2025-06-24,...,78.369754,11178.1,21.630246,49.783013,1-2 Years,35015.205663,45634.794337,56.583750,Medium,61.875776
3,ASSET_4,25,F,-1,NREGI,40000,N,MEDIUM RISK,2025-10-24,2026-04-30,...,74.384431,16874.0,25.615569,40.606061,0-1 Year,38754.405520,43745.594480,53.024963,Low,51.458620
4,ASSET_5,59,F,653,SAL,45000,N,MEDIUM RISK,2024-09-20,2025-05-30,...,69.099157,20571.0,30.900843,44.242424,0-1 Year,40817.718654,41682.281346,50.523977,Low,89.466852


In [20]:
def get_asset_summary(asset_id):

    row = df[df["Agmt Id"] == asset_id].iloc[0]

    summary = f"""
Agreement ID: {row['Agmt Id']}

Customer Age: {row['Cust Age']}

Customer CIBIL: {row['Cust Cibil Score']}

Employment Type: {row['Cust Employment Type']}

Asset Model: {row['Asset Model']}

Loan Amount: ₹{row['Loan Amount']:,.0f}

Residual Value Forecast: ₹{row['Residual_Value_Forecast']:,.0f}

Residual Risk Score: {row['Residual_Risk_Score']:.2f}

Risk Band: {row['Risk_Band']}

Profitability Score: {row['Profitability_Score']:.2f}

Asset Health Index: {row['Asset_Health_Index']:.2f}
"""

    return summary

In [19]:
print(df.columns.tolist())

['Agmt Id', 'Cust Age', 'Cust Gender', 'Cust Cibil Score', 'Cust Employment Type', 'Cust Net Salary', 'Coborrower Flag', 'App Score Risk', 'Agmt Date', 'Seizure Date', 'Sold Date', 'Tenure', 'Cust Net IRR', 'Sold Date Month', 'Sold Date Year', 'Seizure Date Month', 'Seizure Date Year', 'Cust Branch', 'Cust Region', 'Cust State', 'Pincode Tier', 'RC Availability', 'Registration Flag', 'Asset Disc Flag', 'Asset Alloy Flag', 'Asset Variant', 'Asset Model', 'Asset Fuel Type', 'Asset Cost At Disbursal', 'Loan Amount', 'LTV', 'OS Balance At Liquidation', 'Asset Age Months At Seizure', 'Months Spent In Yard', 'Asset Bodycondition', 'Asset Tyrecondition', 'Asset Generalcondition', 'Asset Enginecondition', 'Asset Accident Flag', 'Traiffic Challan Amount', 'Target Sold Amount At Liquidation', 'Asset_Health_Index', 'Recovery_Efficiency_Index', 'LGD', 'LGD_pct', 'Depreciation_Pct', 'Asset_Age_Group', 'Residual_Value_Forecast', 'Residual_Loss', 'Residual_Risk_Score', 'Risk_Band', 'Profitability_Sco

In [21]:
asset_id = df["Agmt Id"].iloc[0]

summary = get_asset_summary(asset_id)

print(summary)


Agreement ID: ASSET_1

Customer Age: 29

Customer CIBIL: -1

Employment Type: NREGI

Asset Model: MOPEDS

Loan Amount: ₹68,712

Residual Value Forecast: ₹31,061

Residual Risk Score: 61.65

Risk Band: High

Profitability Score: 49.15

Asset Health Index: 100.00



In [22]:
prompt = f"""
You are a Senior TVS Credit Risk Officer.

Analyze the following loan application.

{summary}

Provide:

1. Overall Risk Assessment

2. Key Risk Drivers

3. Profitability Outlook

4. Recommended Lending Strategy

5. Whether TVS should:
   - Approve
   - Approve with conditions
   - Reject

6. Executive Summary

Keep response professional.
"""

In [36]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)



In [39]:
response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {
            "role": "system",
            "content": "You are a senior TVS Credit Risk Officer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

**TVS Credit Risk Assessment – Agreement ID: ASSET_1**  
*Date: 5 Sep 2026*  

| Parameter | Value | Interpretation |
|-----------|-------|----------------|
| **Customer Age** | 29 yrs | Young applicant – high earning potential but limited credit history |
| **Credit Score (CIBIL)** | –1 | No score / negative credit record → high default risk |
| **Employment** | NREGI (Non‑regular) | No fixed salary → cash‑flow volatility |
| **Asset** | MOPEDS | Low‑value motorised scooter; high residual risk |
| **Loan Amount** | ₹68,712 | 100 % of vehicle value |
| **Residual Value Forecast** | ₹31,061 | 45 % of loan value at end‑of‑term |
| **Residual Risk Score** | 61.65 | Above threshold (≥60) → high probability of loss |
| **Risk Band** | High | Combined credit‑employment‑asset profile |
| **Profitability Score** | 49.15 | Mid‑range; requires higher margin to compensate risk |
| **Asset Health Index** | 100.00 | Vehicle in perfect condition → minimal operational risk |

---

### 1. Overall Risk

In [42]:
ai_recommendation = response.choices[0].message.content

df.loc[
    df["Agmt Id"] == asset_id,
    "AI_Recommendation"
] = ai_recommendation

In [43]:
results = []

for asset_id in df["Agmt Id"].head(20):

    prompt = get_asset_summary(asset_id)

    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {
                "role": "system",
                "content": "You are a senior TVS Credit Risk Officer."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    recommendation = response.choices[0].message.content

    results.append({
        "Agmt_Id": asset_id,
        "AI_Recommendation": recommendation
    })

print("Completed")

Completed


In [45]:
ai_results = pd.DataFrame(results)

ai_results.head()

,Agmt_Id,AI_Recommendation
0,ASSET_1,**Credit Decision – Asset ID: ASSET_1** \n**P...
1,ASSET_2,**Senior Credit Risk Officer – TVS Credit Risk...
2,ASSET_3,**Senior TVS Credit Risk Officer – Decision & ...
3,ASSET_4,**Senior TVS Credit Risk Officer – Decision & ...
4,ASSET_5,**Credit Risk Assessment – Agreement ID ASSET_...


In [46]:
ai_results.to_csv(
    "../data/processed/ai_credit_recommendations.csv",
    index=False
)

print("Saved")

Saved


In [47]:
portfolio_summary = f"""
Portfolio Size: {len(df)}

Average Risk Score:
{df['Residual_Risk_Score'].mean():.2f}

Average Profitability Score:
{df['Profitability_Score'].mean():.2f}

High Risk Accounts:
{(df['Risk_Band']=='High').sum()}

Medium Risk Accounts:
{(df['Risk_Band']=='Medium').sum()}

Low Risk Accounts:
{(df['Risk_Band']=='Low').sum()}
"""

print(portfolio_summary)


Portfolio Size: 15000

Average Risk Score:
60.68

Average Profitability Score:
50.25

High Risk Accounts:
3750

Medium Risk Accounts:
3750

Low Risk Accounts:
3750



In [50]:
portfolio_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "You are the Chief Risk Officer of TVS Credit."
        },
        {
            "role": "user",
            "content": f"""
Portfolio Summary:

{portfolio_summary}

Provide:

1. Portfolio Health Assessment

2. Major Risks

3. Profitability Outlook

4. Recommended Actions

5. Executive Recommendation
"""
        }
    ]
)

print(portfolio_response.choices[0].message.content)

**Chief Risk Officer – TVS Credit**  
*Portfolio Health Report – 15,000 Accounts*  

| Metric | Value |
|--------|-------|
| **Portfolio Size** | 15,000 |
| **Average Risk Score** | **60.68** |
| **Average Profitability Score** | **50.25** |
| **High‑Risk Accounts** | 3,750 (25 %) |
| **Medium‑Risk Accounts** | 3,750 (25 %) |
| **Low‑Risk Accounts** | 3,750 (25 %) |

> *Risk score interpretation (0 – 100):*  
> **0‑40** = Low risk  
> **41‑70** = Medium risk  
> **71‑100** = High risk  

---

## 1. Portfolio Health Assessment

| Dimension | Current Status | Trend/Implication |
|-----------|----------------|-------------------|
| **Risk Profile** | **Average 60.68** – borderline medium/high risk | 25 % of accounts are high risk. The risk exposure is higher than typical industry benchmarks for a diversified consumer credit portfolio. |
| **Profitability** | **Average 50.25** – moderate return | While returns are acceptable, the risk‑adjusted return (Sharpe‑like) is likely below target, g

In [52]:
def tvs_copilot(asset_id):

    prompt = get_asset_summary(asset_id)

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a senior TVS Credit Risk Officer."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    print("\n" + "="*80)
    print(f"TVS AI CREDIT DECISION REPORT : {asset_id}")
    print("="*80 + "\n")

    print(response.choices[0].message.content)

    return response.choices[0].message.content

In [53]:
asset_id = df["Agmt Id"].iloc[0]

tvs_copilot(asset_id)


TVS AI CREDIT DECISION REPORT : ASSET_1

**Senior TVS Credit Risk Officer – Decision Memo**

---

### 1.  Quick Reference Snapshot

| Parameter | Value | Benchmark / Note |
|-----------|-------|------------------|
| **Agreement ID** | ASSET_1 | – |
| **Customer Age** | 29 yrs | Younger borrowers – higher propensity for higher risk |
| **CIBIL Score** | **–1** | Negative/No credit history → default indicator |
| **Employment Type** | NREGI (Non‑Regular) | Lacks stable income stream |
| **Asset Model** | MOPEDS | High depreciation, lower residual value |
| **Loan Amount** | ₹68,712 |  |
| **Residual Value Forecast** | ₹31,061 | ~45% of original loan – high loss‑given‑default risk |
| **Residual Risk Score** | **61.65** | High risk (scale 0–100) |
| **Risk Band** | **High** | Classifies the exposure as “High” per TVS policy |
| **Profitability Score** | 49.15 | Moderate – below desired threshold (≥ 70) |
| **Asset Health Index** | 100.00 | Asset is in perfect condition (no wear‑and‑tear)

'**Senior TVS Credit Risk Officer – Decision Memo**\n\n---\n\n### 1.  Quick Reference Snapshot\n\n| Parameter | Value | Benchmark / Note |\n|-----------|-------|------------------|\n| **Agreement ID** | ASSET_1 | – |\n| **Customer Age** | 29\u202fyrs | Younger borrowers – higher propensity for higher risk |\n| **CIBIL Score** | **–1** | Negative/No credit history → default indicator |\n| **Employment Type** | NREGI (Non‑Regular) | Lacks stable income stream |\n| **Asset Model** | MOPEDS | High depreciation, lower residual value |\n| **Loan Amount** | ₹68,712 |  |\n| **Residual Value Forecast** | ₹31,061 | ~45% of original loan – high loss‑given‑default risk |\n| **Residual Risk Score** | **61.65** | High risk (scale 0–100) |\n| **Risk Band** | **High** | Classifies the exposure as “High” per TVS policy |\n| **Profitability Score** | 49.15 | Moderate – below desired threshold (≥\u202f70) |\n| **Asset Health Index** | 100.00 | Asset is in perfect condition (no wear‑and‑tear) |\n\n---\n\n